# AI Fraud Detection and Adversarial Robustness
## PHASE 1 — Fraud Detection
Run the following Phase 1 cells in order. Later phases remain placeholders.

### Cell 1 — Environment / Imports

In [ ]:
from pathlib import Path
import json
import os
project_candidates = [Path.cwd(), Path('/content/AI_Fraud_Adversarial')]
PROJECT_ROOT = next((p for p in project_candidates if (p / 'requirements-colab.txt').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Clone the GitHub repository to /content/AI_Fraud_Adversarial, then rerun this cell.')
os.chdir(PROJECT_ROOT)
%pip install -q --upgrade -r requirements-colab.txt
import joblib
import matplotlib.pyplot as plt
import pandas as pd


### Cell 2 — Google Drive Mount

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Cell 3 — Configuration

In [ ]:
DATASET_PATH = Path('/content/drive/MyDrive/AI_Fraud_Adversarial/data/paysim.csv')
DEVELOPMENT_MODE = True  # Set False only for the final full-data run.
RUN_MODE = 'DEVELOPMENT_MODE' if DEVELOPMENT_MODE else 'FULL_MODE'
RANDOM_SEED = 42
OUTPUT_DIR = Path('/content/drive/MyDrive/AI_Fraud_Adversarial/outputs')
FIGURES_DIR = OUTPUT_DIR / 'figures' / 'phase1'
MODELS_DIR = OUTPUT_DIR / 'models'
METRICS_DIR = OUTPUT_DIR / 'metrics'
for directory in (FIGURES_DIR, MODELS_DIR, METRICS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print({'run_mode': RUN_MODE, 'dataset': str(DATASET_PATH), 'outputs': str(OUTPUT_DIR)})

### Cell 4 — Dataset Check

In [ ]:
if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'PaySim CSV not found: {DATASET_PATH}')
print(f'PaySim found: {DATASET_PATH} ({DATASET_PATH.stat().st_size / 1e9:.2f} GB)')

### Cell 5 — Load PaySim

In [ ]:
from src.data_loader import load_paysim
data = load_paysim(DATASET_PATH, run_mode=RUN_MODE, random_seed=RANDOM_SEED)
print(f'Loaded {len(data):,} rows in {RUN_MODE}.')

### Cell 6 — Inspect Dataset

In [ ]:
from src.data_loader import summarize_paysim
summary = summarize_paysim(data)
print('Shape:', summary['shape'])
print('Columns:', summary['columns'])
print('Missing values:', summary['missing_values'])
print('Transaction types:', summary['transaction_types'])
print('Class counts:', summary['class_counts'])
print(f"Fraud percentage: {summary['fraud_percentage']:.6f}%")

### Cell 7 — Preprocess and Engineer Features

In [ ]:
from src.preprocessing import prepare_features_and_target
X, y = prepare_features_and_target(data)
print('Raw model inputs:', X.columns.tolist())
print('Target counts:', y.value_counts().sort_index().to_dict())

### Cell 8 — Train/Test Split

In [ ]:
from src.preprocessing import fit_transform_train_test
from src.train import stratified_train_test_split
X_train_raw, X_test_raw, y_train, y_test = stratified_train_test_split(
    X, y, random_state=RANDOM_SEED
)
X_train, X_test, preprocessor = fit_transform_train_test(X_train_raw, X_test_raw)
feature_names = X_train.columns.tolist()
print('Training counts:', y_train.value_counts().sort_index().to_dict())
print('Untouched test counts:', y_test.value_counts().sort_index().to_dict())
print('Encoded features:', feature_names)

### Cell 9 — SMOTE

In [ ]:
from src.config import SMOTE_SAMPLING_STRATEGY
from src.preprocessing import resample_training_data
print('Before SMOTE:', y_train.value_counts().sort_index().to_dict())
X_train_smote, y_train_smote = resample_training_data(
    X_train, y_train, random_state=RANDOM_SEED,
    sampling_strategy=SMOTE_SAMPLING_STRATEGY
)
print('After SMOTE:', pd.Series(y_train_smote).value_counts().sort_index().to_dict())
print('Test data was not passed to SMOTE.')

### Cell 10 — Train Baseline Model

In [ ]:
from src.train import train_baseline_model
baseline_model = train_baseline_model(X_train_smote, y_train_smote)
print('Baseline XGBoost training complete.')

### Cell 11 — Evaluate Baseline

In [ ]:
from src.evaluate import evaluate_binary_classifier
from src.visualization import (
    plot_confusion_matrix, plot_original_class_distribution,
    plot_precision_recall, plot_smote_distributions,
)
evaluation = evaluate_binary_classifier(baseline_model, X_test, y_test)
print(json.dumps(evaluation['metrics'], indent=2))
print('Fraud Recall (primary concern):', evaluation['metrics']['recall'])
print(pd.DataFrame(evaluation['classification_report']).transpose())
plot_original_class_distribution(y, FIGURES_DIR)
plot_smote_distributions(y_train, y_train_smote, FIGURES_DIR)
plot_confusion_matrix(evaluation['confusion_matrix'], FIGURES_DIR)
plot_precision_recall(evaluation, FIGURES_DIR)
plt.show()

### Cell 12 — Save Artifacts

In [ ]:
from src.evaluate import save_metrics
baseline_model_path = MODELS_DIR / 'baseline_model.joblib'
feature_names_path = MODELS_DIR / 'feature_names.joblib'
preprocessor_path = MODELS_DIR / 'baseline_preprocessor.joblib'
metrics_path = METRICS_DIR / 'phase1_baseline_metrics.json'
joblib.dump(baseline_model, baseline_model_path)
joblib.dump(feature_names, feature_names_path)
joblib.dump(preprocessor, preprocessor_path)
save_metrics(evaluation, metrics_path)
print('Saved:', baseline_model_path, feature_names_path, preprocessor_path, metrics_path, sep='\n- ')

# PHASE 2 — SHAP Explainability
Reuse the trained Phase 1 baseline. Do not retrain it.

### Cell 13 — Load Phase 1 Artifacts

In [ ]:
from src.explainability import load_phase1_artifacts
baseline_model, saved_preprocessor, saved_feature_names = load_phase1_artifacts(
    MODELS_DIR / 'baseline_model.joblib',
    MODELS_DIR / 'baseline_preprocessor.joblib',
    MODELS_DIR / 'feature_names.joblib',
)
print(f'Loaded baseline model with {len(saved_feature_names)} encoded features.')
print('The baseline model was loaded, not retrained.')

### Cell 14 — Prepare SHAP Evaluation Sample

In [ ]:
from src.explainability import prepare_shap_evaluation_sample, transform_with_saved_preprocessor
# Reuse the live Phase 1 test split. Reconstruct it deterministically after a restart.
if 'X_test' not in globals() or 'y_test' not in globals():
    from src.data_loader import load_paysim
    from src.preprocessing import prepare_features_and_target
    from src.train import stratified_train_test_split
    reconstructed_data = load_paysim(DATASET_PATH, run_mode=RUN_MODE, random_seed=RANDOM_SEED)
    reconstructed_X, reconstructed_y = prepare_features_and_target(reconstructed_data)
    _, X_test_raw, _, y_test = stratified_train_test_split(
        reconstructed_X, reconstructed_y, random_state=RANDOM_SEED
    )
    X_test = transform_with_saved_preprocessor(
        saved_preprocessor, X_test_raw, saved_feature_names
    )
else:
    if X_test.columns.tolist() != saved_feature_names:
        raise ValueError('Live test features do not match saved Phase 1 feature names.')
X_shap, y_shap = prepare_shap_evaluation_sample(
    X_test, y_test, random_state=RANDOM_SEED
)
print('SHAP sample shape:', X_shap.shape)
print('SHAP sample labels:', y_shap.value_counts().sort_index().to_dict())
print('This class-aware held-out sample is for explanation, not prevalence estimation.')

### Cell 15 — Create SHAP Explainer

In [ ]:
from src.explainability import create_tree_explainer, compute_shap_values
shap_explainer = create_tree_explainer(baseline_model)
shap_values = compute_shap_values(shap_explainer, X_shap)
print('Computed Tree SHAP values:', shap_values.values.shape)
print('Positive SHAP values push the raw model margin toward fraud; negative values push toward legitimate.')

### Cell 16 — Global SHAP Feature Importance

In [ ]:
from src.explainability import rank_global_importance
global_importance = rank_global_importance(shap_values)
display(global_importance)
import shap
shap.plots.bar(shap_values, max_display=15)

### Cell 17 — SHAP Summary Plot

In [ ]:
shap.plots.beeswarm(shap_values, max_display=15)

### Cell 18 — Explain Correctly Detected Fraud

In [ ]:
from src.explainability import build_interpretation, explain_selected_row, select_correct_examples
selected_examples = select_correct_examples(
    baseline_model, X_test, y_test, random_state=RANDOM_SEED
)
fraud_explanation, fraud_table = explain_selected_row(
    shap_explainer, X_test, selected_examples['fraud']
)
fraud_interpretation = build_interpretation(
    selected_examples['fraud'], fraud_table
)
print(json.dumps(fraud_interpretation, indent=2, default=str))
display(fraud_table.head(15))
shap.plots.waterfall(fraud_explanation, max_display=15)

### Cell 19 — Explain Correctly Detected Legitimate Transaction

In [ ]:
legitimate_explanation, legitimate_table = explain_selected_row(
    shap_explainer, X_test, selected_examples['legitimate']
)
legitimate_interpretation = build_interpretation(
    selected_examples['legitimate'], legitimate_table
)
print(json.dumps(legitimate_interpretation, indent=2, default=str))
display(legitimate_table.head(15))
shap.plots.waterfall(legitimate_explanation, max_display=15)

### Cell 20 — Save SHAP Outputs and Interpretation

In [ ]:
from src.explainability import save_global_plots, save_phase2_outputs, save_waterfall_plot
SHAP_OUTPUT_DIR = OUTPUT_DIR / 'shap'
save_global_plots(shap_values, SHAP_OUTPUT_DIR, max_display=15)
save_waterfall_plot(
    fraud_explanation, SHAP_OUTPUT_DIR / 'correct_fraud_waterfall.png', max_display=15
)
save_waterfall_plot(
    legitimate_explanation, SHAP_OUTPUT_DIR / 'correct_legitimate_waterfall.png', max_display=15
)
saved_shap_paths = save_phase2_outputs(
    SHAP_OUTPUT_DIR,
    global_importance,
    {'fraud': fraud_table, 'legitimate': legitimate_table},
    {'fraud': fraud_interpretation, 'legitimate': legitimate_interpretation},
    y_shap,
)
print('Saved Phase 2 outputs:')
for path in sorted(SHAP_OUTPUT_DIR.iterdir()):
    print('-', path)

# PHASE 3 — Adversarial Evasion Attack
Targeted, decision-based HopSkipJump on correctly detected held-out fraud only. Generated test samples are evaluation-only and must never be used for training.

### Cell 21 — ART Setup and Compatibility Check

In [ ]:
import importlib.util
import subprocess
import sys
if importlib.util.find_spec('art') is None:
    print('ART is missing; installing adversarial-robustness-toolbox from requirements-colab.txt...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'])
import art
from art.attacks.evasion import HopSkipJump
from art.estimators.classification import BlackBoxClassifier, XGBoostClassifier
from src.attack import verify_art_compatibility
art_compatibility = verify_art_compatibility(baseline_model, saved_feature_names)
print(json.dumps(art_compatibility, indent=2))
print('Available ART wrappers:', BlackBoxClassifier.__name__, XGBoostClassifier.__name__)
print('Selected attack:', HopSkipJump.__name__)
print('Reason: HSJ needs predictions/decisions, not gradients, and the black-box adapter enforces valid feature dependencies.')

### Cell 22 — Build Feature Threat Model

In [ ]:
from src.attack import build_feature_threat_model
from src.config import ATTACK_RELATIVE_BOUND
feature_threat_model = build_feature_threat_model(
    X_test, relative_bound=ATTACK_RELATIVE_BOUND
)
display(feature_threat_model)
print('Direct monetary changes are bounded and non-negative.')
print('Engineered values are recomputed; step and one-hot transaction type are protected.')

### Cell 23 — Select Correctly Detected Fraud Test Samples

In [ ]:
from src.attack import select_correctly_detected_test_fraud
from src.config import ATTACK_SAMPLE_SIZE
X_attack_clean, y_attack, attack_population = select_correctly_detected_test_fraud(
    baseline_model, X_test, y_test, sample_size=ATTACK_SAMPLE_SIZE,
    random_state=RANDOM_SEED,
)
print(json.dumps(attack_population, indent=2))
print('Selected source test indices:', X_attack_clean.index.tolist())
assert y_attack.eq(1).all()
assert (baseline_model.predict(X_attack_clean) == 1).all()

### Cell 24 — Configure Adversarial Attack

In [ ]:
from src.config import (ATTACK_INIT_EVAL, ATTACK_INIT_SIZE, ATTACK_MAX_EVAL, ATTACK_MAX_ITER)
attack_configuration = {
    'attack': 'Targeted HopSkipJump',
    'target_class': 0,
    'relative_monetary_bound': ATTACK_RELATIVE_BOUND,
    'max_iter': ATTACK_MAX_ITER,
    'max_eval': ATTACK_MAX_EVAL,
    'init_eval': ATTACK_INIT_EVAL,
    'init_size': ATTACK_INIT_SIZE,
    'random_seed': RANDOM_SEED,
    'evaluation_only': True,
}
print(json.dumps(attack_configuration, indent=2))

### Cell 25 — Generate Adversarial TEST Samples

In [ ]:
from src.attack import generate_constrained_hopskipjump
X_attack_adversarial, attack_execution = generate_constrained_hopskipjump(
    baseline_model, X_attack_clean,
    relative_bound=ATTACK_RELATIVE_BOUND,
    max_iter=ATTACK_MAX_ITER, max_eval=ATTACK_MAX_EVAL,
    init_eval=ATTACK_INIT_EVAL, init_size=ATTACK_INIT_SIZE,
    random_state=RANDOM_SEED, verbose=True,
)
print(json.dumps(attack_execution, indent=2))
print('Generated adversarial test matrix:', X_attack_adversarial.shape)
print('Original X_test remains unchanged; adversarial rows are stored separately.')

### Cell 26 — Evaluate Recall Under Attack

In [ ]:
from src.evaluate import evaluate_adversarial_evasion
phase3_metrics, attacked_samples = evaluate_adversarial_evasion(
    baseline_model, X_attack_clean, X_attack_adversarial,
    full_test_clean_fraud_recall=attack_population['full_test_clean_fraud_recall'],
    attack_metadata=attack_execution,
)
print('Full-test clean fraud recall:', phase3_metrics['full_test_clean_fraud_recall'])
print('Selected-sample clean recall:', phase3_metrics['attacked_sample_clean_recall'])
print('Recall under attack:', phase3_metrics['recall_under_attack'])
print('Recall drop:', phase3_metrics['recall_drop_on_attacked_sample'])

### Cell 27 — Calculate Attack Success Rate

In [ ]:
from src.visualization import plot_attack_perturbations, plot_attack_probabilities
PHASE3_FIGURES_DIR = OUTPUT_DIR / 'figures' / 'phase3'
print('Attack success definition:', phase3_metrics['attack_success_definition'])
print('Successful evasions:', phase3_metrics['successful_evasions'])
print('Number attacked:', phase3_metrics['number_attacked'])
print('Attack success/evasion rate:', phase3_metrics['attack_success_rate'])
print('Average L2 perturbation:', phase3_metrics['average_l2_perturbation'])
print('Maximum L2 perturbation:', phase3_metrics['max_l2_perturbation'])
display(attacked_samples)
plot_attack_probabilities(attacked_samples, PHASE3_FIGURES_DIR)
plot_attack_perturbations(attacked_samples, PHASE3_FIGURES_DIR)
plt.show()

### Cell 28 — Save Phase 3 Attack Outputs

In [ ]:
from src.evaluate import save_attack_outputs
phase3_metrics_path = METRICS_DIR / 'phase3_attack_metrics.json'
phase3_samples_path = METRICS_DIR / 'phase3_attacked_samples.csv'
phase3_threat_model_path = METRICS_DIR / 'phase3_feature_constraints.csv'
save_attack_outputs(
    phase3_metrics, attacked_samples, phase3_metrics_path, phase3_samples_path
)
feature_threat_model.to_csv(phase3_threat_model_path, index=False)
print('Saved Phase 3 outputs:')
for path in (phase3_metrics_path, phase3_samples_path, phase3_threat_model_path):
    print('-', path)
for path in sorted(PHASE3_FIGURES_DIR.iterdir()):
    print('-', path)
print('Evaluation-only adversarial test samples were not saved as training data.')

# PHASE 4 — Multiple Attack Comparison
Not implemented.

# PHASE 5 — Model Hardening
Not implemented.

# PHASE 6 — Streamlit Dashboard
Not implemented.

# PHASE 7 — Real-Time Transaction Simulation
Not implemented.

# PHASE 8 — Concept Drift
Not implemented.

# FINAL — Results and Conclusions
Complete after all phases have actual execution results.